# Build a planner, worker, writer multi-agent solution with Microsoft Agent Framework and observe the handoff

Your research team is tired of asking a single chat model open-ended questions like "what are the trade-offs of X versus Y in production" and getting a wall of text that mixes hand-waved opinions with plausible-sounding citations. The lead researcher wants a system where one agent plans the subtasks, separate workers execute each, and a writer consolidates the results with explicit per-worker citations so the team can audit each subtask in isolation. You have one session to build the three-agent solution with Microsoft Agent Framework 1.0, run it on a single research prompt, and walk the trace to prove the handoffs happened as designed.

What you will build:

- A Foundry hub plus project in `eastus`, plus a single `gpt-4o-mini` deployment shared by all three agents.
- Three Agent Framework 1.0 agents: `Planner` (decomposes a question into 2 to 4 subtasks), `Worker` (executes one subtask against a stub `web_search` tool), and `Writer` (consolidates with per-worker citations).
- A sequential orchestration: Planner output flows into the Worker pool, Worker outputs flow into the Writer.
- A run on a seeded research prompt with full trace capture.
- A per-agent token-usage breakdown so the cost story is concrete.

**Time.** Plan for about 75 minutes hands-on. Foundry provisioning is 5 to 8 minutes; agent declarations and orchestration are fast; the run is the slow part because each agent's context grows. Budget up to 120 minutes total.

**Cost.** Multi-agent flows accumulate tokens because each agent's context grows: Planner gets the question, Worker gets the question plus a subtask, Writer gets the question plus all the Worker outputs. The token math compounds. A clean run lands under one cent; heavy debugging across all three agents tops out around 50 cents. The lab clean run is still cents; what you are paying for is the muscle memory to NOT scale this to 10 agents without thinking about the token bill.

This lab maps to AI-103 Domain 2: Implement generative AI and agentic solutions (33% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs and Microsoft Agent Framework 1.0.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents.
# - agent-framework>=1.0.0 is the April 3 2026 GA release that merges
#   Semantic Kernel and AutoGen. Tutorials still reference the old packages;
#   do NOT install semantic-kernel or autogen for this lab per the cert YAML
#   risks list.

!pip install --quiet labrun-checks==0.6.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 agent-framework>=1.0.0 openai>=2.0.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.

import atexit
import getpass
import json
import time

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from openai import AzureOpenAI

from agent_framework import Agent, FunctionTool, SequentialOrchestration

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "multi-agent-orchestration"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

# Resolved during setup.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
AZURE_CREDENTIAL = None

# State populated by tasks for downstream checkpoints.
PLANNER_AGENT = None
WORKER_AGENT = None
WRITER_AGENT = None
ORCHESTRATION = None
RUN_RESULT = None
USAGE_BY_AGENT = {"Planner": [], "Worker": [], "Writer": []}

# Pricing for the wallet check.
PRICE_PER_1M_INPUT_USD = 0.15
PRICE_PER_1M_OUTPUT_USD = 0.60
COST_CEILING_USD = 0.50

RESEARCH_PROMPT = (
    "What are the trade-offs of vector-only versus hybrid retrieval in production RAG?"
)

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# No critical resources: agents are application-side abstractions in this
# lab, and the GPT-4o-mini Standard deployment does not bill at zero traffic.
# Cleanup tears every resource down for orphan-scan hygiene.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up Foundry, deploy GPT-4o-mini, and declare three Agent Framework 1.0 agents

The Foundry stack is the same shape as Lab 01. Once it is up, you declare three agents against the same deployment using `agent_framework.Agent`. Each agent gets a name, a system prompt that defines its role, and (for the Worker) a `web_search` tool.

Build it in this order:

1. Create the resource group, hub, and project. Discover the attached Azure OpenAI account. Deploy `gpt-4o-mini` at Standard SKU.
2. Build the `web_search` stub: a local Python function that returns mock results for two seeded queries (vector retrieval trade-offs, hybrid retrieval trade-offs) and a "no results" placeholder otherwise. Wrap it in `FunctionTool(web_search, name="web_search")`.
3. Declare three agents:
   - `Planner`: instructions tell it to decompose the user research question into 2 to 4 subtasks. No tools.
   - `Worker`: instructions tell it to take one subtask and call `web_search` at least once, then summarise the findings. Tools: `[web_search_tool]`.
   - `Writer`: instructions tell it to consolidate Worker outputs into a single paragraph with `worker_id` citations. No tools.
4. Pin every agent to `model=DEPLOYMENT_NAME` so the cost story is consistent across all three.

**Anti-pattern: the deprecated Semantic Kernel and AutoGen packages.** Microsoft Agent Framework 1.0 (April 3, 2026) merged both into a single API surface. The checkpoint REJECTS imports from `semantic_kernel.agents` and from `autogen.agents`; both are deprecated for new projects per the cert YAML risks list.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision Foundry, deploy GPT-4o-mini, declare three agents.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()

for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the model deployment go through Succeeded purgatory, about 1 to 2 minutes...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Build the stub web_search tool. Deterministic mock data for two seeded
# queries, "no results" placeholder otherwise. Returning a dict gives the
# Worker a structured result to summarise.
MOCK_SEARCH = {
    "vector retrieval trade-offs": [
        {
            "title": "Vector-only retrieval pitfalls",
            "snippet": "Vector retrieval excels at semantic match but underperforms on "
                       "exact-string queries like SKU codes; the top-k ordering can be "
                       "noisy without a reranker.",
        },
        {
            "title": "When pure-vector is the right call",
            "snippet": "For long-form natural-language queries where exact-match is not "
                       "the priority, pure-vector retrieval is the cheapest path.",
        },
    ],
    "hybrid retrieval trade-offs": [
        {
            "title": "Hybrid retrieval cost-benefit",
            "snippet": "Hybrid (vector plus BM25) covers exact-match and semantic in one "
                       "query; the extra BM25 cost is negligible compared to model inference.",
        },
        {
            "title": "Hybrid plus semantic ranker for production",
            "snippet": "Adding the semantic ranker over hybrid stabilises top-k ordering "
                       "and is the AI-103-documented default for production RAG.",
        },
    ],
}


def web_search(query: str) -> dict:
    """Stub web search. Returns mock results for two seeded queries."""
    key = query.lower()
    for seed in MOCK_SEARCH:
        if seed in key or key in seed:
            return {"query": query, "results": MOCK_SEARCH[seed]}
    return {"query": query, "results": []}


web_search_tool = FunctionTool(web_search, name="web_search")

# Construct an AzureOpenAI client to back every agent. DefaultAzureCredential
# converts into a bearer token provider per the AOAI SDK contract.
token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
backing_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

planner_instructions = (
    "You are the Planner. Decompose the user's research question into 2 to 4 "
    "concrete subtasks, each phrased as a search query. Return JSON exactly in "
    'this shape: {"subtasks": ["query1", "query2", ...]}. Cap at 4 subtasks. '
    "Do not call any tools."
)
worker_instructions = (
    "You are the Worker. Take exactly one subtask as input, call web_search at "
    "least once with the subtask as the query, then summarise the findings in "
    "two sentences. Return JSON exactly in this shape: "
    '{"worker_id": "<unique id>", "subtask": "<input>", "summary": "<two sentences>"}. '
    "Always call web_search; do not answer from prior knowledge."
)
writer_instructions = (
    "You are the Writer. Consolidate the Worker outputs (provided as a list of "
    'JSON objects with worker_id, subtask, summary) into a single paragraph. '
    "Cite every worker_id explicitly in the paragraph (for example, "
    "'[worker_id=W1] notes that ...'). Return JSON exactly in this shape: "
    '{"paragraph": "<one paragraph with [worker_id=...] citations>", '
    '"cited_worker_ids": ["W1", "W2", ...]}.'
)

# YOUR CODE: Declare the three agents. Pass model=DEPLOYMENT_NAME on each.
# PLANNER_AGENT = Agent(name="Planner", instructions=planner_instructions,
#                       model=DEPLOYMENT_NAME, client=backing_client)
# WORKER_AGENT = Agent(name="Worker", instructions=worker_instructions,
#                      model=DEPLOYMENT_NAME, client=backing_client,
#                      tools=[web_search_tool])
# WRITER_AGENT = Agent(name="Writer", instructions=writer_instructions,
#                      model=DEPLOYMENT_NAME, client=backing_client)

print(f"Planner declared: {PLANNER_AGENT.name}")
print(f"Worker declared: {WORKER_AGENT.name} (tools: {[t.name for t in WORKER_AGENT.tools or []]})")
print(f"Writer declared: {WRITER_AGENT.name}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Three agents declared with the Agent Framework 1.0 API.
# Each has the correct name, references DEPLOYMENT_NAME, and the Worker
# has one function-type tool named web_search. Rejects imports traceable
# to semantic_kernel.agents or autogen.agents.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        # Reject any agent whose class module path traces back to the
        # deprecated surfaces.
        for label, agent in (("Planner", PLANNER_AGENT), ("Worker", WORKER_AGENT), ("Writer", WRITER_AGENT)):
            if agent is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{label} agent is None. Did Task 1 finish declaring all three?",
                )
            module = type(agent).__module__ or ""
            if module.startswith("semantic_kernel") or module.startswith("autogen"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{label} agent class lives in {module!r}, which is the deprecated "
                        f"Semantic Kernel or AutoGen surface. Microsoft Agent Framework 1.0 "
                        f"replaces both as of April 2026. Import Agent from agent_framework."
                    ),
                )
            if not module.startswith("agent_framework"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{label} agent class lives in {module!r}, expected agent_framework.*"
                    ),
                )

        # Per-agent attribute checks.
        if PLANNER_AGENT.name != "Planner":
            return CheckpointResult(
                status="fail",
                error_reason=f"Planner agent name is {PLANNER_AGENT.name!r}, expected 'Planner'.",
            )
        if WORKER_AGENT.name != "Worker":
            return CheckpointResult(
                status="fail",
                error_reason=f"Worker agent name is {WORKER_AGENT.name!r}, expected 'Worker'.",
            )
        if WRITER_AGENT.name != "Writer":
            return CheckpointResult(
                status="fail",
                error_reason=f"Writer agent name is {WRITER_AGENT.name!r}, expected 'Writer'.",
            )

        for label, agent in (("Planner", PLANNER_AGENT), ("Worker", WORKER_AGENT), ("Writer", WRITER_AGENT)):
            model_ref = getattr(agent, "model", None)
            if model_ref != DEPLOYMENT_NAME:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{label} model is {model_ref!r}, expected {DEPLOYMENT_NAME!r}. "
                        f"Pin every agent to the same deployment so the cost story is consistent."
                    ),
                )

        worker_tools = WORKER_AGENT.tools or []
        if len(worker_tools) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Worker has {len(worker_tools)} tools, expected 1 (web_search). "
                    f"Pass tools=[web_search_tool] on the Worker only."
                ),
            )
        if worker_tools[0].name != "web_search":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Worker tool name is {worker_tools[0].name!r}, expected 'web_search'."
                ),
            )

        # Planner and Writer must not have tools.
        if PLANNER_AGENT.tools:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Planner has tools {[t.name for t in PLANNER_AGENT.tools]} but should have none. "
                    f"Only the Worker needs web_search."
                ),
            )
        if WRITER_AGENT.tools:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Writer has tools {[t.name for t in WRITER_AGENT.tools]} but should have none."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which agent failed (Planner, Worker, or Writer) and which attribute is off. If the module path complaint fires, you imported `Agent` from `semantic_kernel.agents` or `autogen.agents`; switch to `from agent_framework import Agent`. If Worker has 0 tools, the `tools=[...]` argument did not land.

</details>

<details><summary>Hint 2 (direction)</summary>

Three constructors. `Agent(name="Planner", instructions=planner_instructions, model=DEPLOYMENT_NAME, client=backing_client)`. Same shape for the Worker (add `tools=[web_search_tool]`) and Writer. The `instructions`, `DEPLOYMENT_NAME`, `backing_client`, and `web_search_tool` variables are already prepared in the cell above. The Writer and Planner have NO tools.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
PLANNER_AGENT = Agent(
    name="Planner",
    instructions=planner_instructions,
    model=DEPLOYMENT_NAME,
    client=backing_client,
)
WORKER_AGENT = Agent(
    name="Worker",
    instructions=worker_instructions,
    model=DEPLOYMENT_NAME,
    client=backing_client,
    tools=[web_search_tool],
)
WRITER_AGENT = Agent(
    name="Writer",
    instructions=writer_instructions,
    model=DEPLOYMENT_NAME,
    client=backing_client,
)
```

</details>

**Wallet check.** Still at $0.00. Declaring agents and the orchestration graph is local Python; no tokens have been spent. Coffee is well ahead.

## Task 2: Wire the sequential orchestration: Planner to Worker pool to Writer

Microsoft Agent Framework's `SequentialOrchestration` runs agents in declared order. Each step receives the previous step's output. The framework handles passing the conversation context and tool results between steps.

Build it in this order:

1. Construct `ORCHESTRATION = SequentialOrchestration(agents=[PLANNER_AGENT, WORKER_AGENT, WRITER_AGENT])`. The order in the list IS the execution order; the validator asserts Planner runs before Worker and Worker before Writer.
2. The Worker step iterates internally: the framework runs the Worker once per subtask the Planner emitted (this is the framework's built-in fan-out for sequential steps that consume list-shaped input from a prior step).
3. The Writer step receives the full Worker output list as input.

**Why sequential, not parallel.** The framework supports `ParallelOrchestration` and `GroupChat` topologies too. The lab uses sequential because the trace is easier to walk and the "exactly once per subtask" assertion in Checkpoint 3 is straightforward. In production with non-overlapping Worker tasks, parallel is the right call.

In [ ]:
# NBVAL_SKIP
# Task 2: Construct the sequential orchestration.

# YOUR CODE: Build the orchestration. Pass the three agents in order:
# ORCHESTRATION = SequentialOrchestration(
#     agents=[PLANNER_AGENT, WORKER_AGENT, WRITER_AGENT],
# )

print(f"Orchestration: {type(ORCHESTRATION).__name__}")
agent_names = [a.name for a in ORCHESTRATION.agents]
print(f"Step order: {agent_names}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Orchestration is the SequentialOrchestration type, contains
# the three agents in order Planner -> Worker -> Writer.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if ORCHESTRATION is None:
            return CheckpointResult(
                status="fail",
                error_reason="ORCHESTRATION is None. Run Task 2 to construct it.",
            )

        if type(ORCHESTRATION).__name__ != "SequentialOrchestration":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Orchestration type is {type(ORCHESTRATION).__name__!r}, expected "
                    f"'SequentialOrchestration'. The lab teaches sequential for trace clarity; "
                    f"ParallelOrchestration can double-fire subtasks and break Checkpoint 3."
                ),
            )

        agents = list(getattr(ORCHESTRATION, "agents", []) or [])
        names = [getattr(a, "name", None) for a in agents]
        expected = ["Planner", "Worker", "Writer"]
        if names != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Orchestration agent order is {names}, expected {expected}. "
                    f"Pass agents=[PLANNER_AGENT, WORKER_AGENT, WRITER_AGENT] in that exact order."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint will tell you if the type is wrong or the order is wrong. If the type complaint fires, you instantiated `ParallelOrchestration` or `GroupChat`; switch to `SequentialOrchestration`. If the order is wrong, your list does not start with the Planner.

</details>

<details><summary>Hint 2 (direction)</summary>

One line: `ORCHESTRATION = SequentialOrchestration(agents=[PLANNER_AGENT, WORKER_AGENT, WRITER_AGENT])`. The three agent globals are already populated from Task 1. The list order IS the execution order.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
ORCHESTRATION = SequentialOrchestration(
    agents=[PLANNER_AGENT, WORKER_AGENT, WRITER_AGENT],
)
```

</details>

**Wallet check.** Still at $0.00. Construction of the orchestration is local; no inference yet. Coffee continues to dominate.

## Task 3: Run the orchestration on the seeded prompt and walk the trace

This is the `code_observe` hybrid: the CODE phase runs the system, the OBSERVE phase reads the trace structure.

Build it in this order:

1. Call `RUN_RESULT = ORCHESTRATION.run(message=RESEARCH_PROMPT, capture_trace=True)`. The framework drives Planner, then fans out to Worker per subtask, then the Writer.
2. The result exposes `RUN_RESULT.trace`, an ordered list of step records. Each record has `agent_name`, `inputs`, `outputs`, `tool_calls`, and `usage` (per-call token counts).
3. Walk the trace and confirm: the Planner step emitted at least 2 subtasks (parse its JSON output); the Worker step appeared exactly N times where N is the number of subtasks; every Worker invocation called `web_search` at least once.
4. Capture per-agent usage into `USAGE_BY_AGENT` for Checkpoint 4.

The Worker output payload is a list of JSON objects in this lab; the Writer prompt expects this exact shape. The framework's `capture_trace=True` flag is what makes the trace inspectable; without it `RUN_RESULT.trace` is `None`.

In [ ]:
# NBVAL_SKIP
# Task 3 (CODE phase): run the orchestration and capture the trace.

print("Planner is thinking about how to slice this question into subtasks...")
# YOUR CODE: Run the orchestration with trace capture. Construct
# RUN_RESULT = ORCHESTRATION.run(message=RESEARCH_PROMPT, capture_trace=True).

print()
print(f"Orchestration finished. Trace has {len(RUN_RESULT.trace)} steps.")
print(f"Final paragraph: {RUN_RESULT.output[:200]}...")

In [ ]:
# NBVAL_SKIP
# Task 3 (OBSERVE phase): walk the trace and capture per-agent usage.
# This cell does not create AWS or Azure resources; it inspects the run.

trace = RUN_RESULT.trace
planner_subtasks = []
worker_invocations = []
worker_tool_calls = []
writer_steps = []
USAGE_BY_AGENT = {"Planner": [], "Worker": [], "Writer": []}

for step in trace:
    name = step.agent_name
    if name == "Planner":
        try:
            parsed = json.loads(step.outputs)
            planner_subtasks = parsed.get("subtasks", []) or []
        except (json.JSONDecodeError, TypeError):
            planner_subtasks = []
    elif name == "Worker":
        worker_invocations.append(step)
        worker_tool_calls.append(step.tool_calls or [])
    elif name == "Writer":
        writer_steps.append(step)
    if step.usage:
        USAGE_BY_AGENT[name].append({
            "prompt_tokens": step.usage.prompt_tokens,
            "completion_tokens": step.usage.completion_tokens,
        })

print("Planner subtasks emitted:")
for i, s in enumerate(planner_subtasks, start=1):
    print(f"  {i}. {s}")

print()
print(f"Worker ran {len(worker_invocations)} times.")
for i, calls in enumerate(worker_tool_calls, start=1):
    print(f"  Worker invocation {i}: {len(calls)} web_search call(s).")

print()
print(f"Writer ran {len(writer_steps)} time(s).")

print()
print("Per-agent token usage:")
for name in ("Planner", "Worker", "Writer"):
    entries = USAGE_BY_AGENT[name]
    in_sum = sum(e["prompt_tokens"] for e in entries)
    out_sum = sum(e["completion_tokens"] for e in entries)
    print(f"  {name}: {len(entries)} call(s), {in_sum} input + {out_sum} output tokens")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Planner emitted >= 2 subtasks. Worker ran exactly that many
# times. Every Worker invocation called web_search at least once. Writer ran
# exactly once and its paragraph cites every worker_id from the worker outputs.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if RUN_RESULT is None or not getattr(RUN_RESULT, "trace", None):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "RUN_RESULT.trace is missing or empty. Re-run with capture_trace=True."
                ),
            )

        trace = RUN_RESULT.trace
        planner_outputs = [s.outputs for s in trace if s.agent_name == "Planner"]
        worker_steps = [s for s in trace if s.agent_name == "Worker"]
        writer_steps = [s for s in trace if s.agent_name == "Writer"]

        if not planner_outputs:
            return CheckpointResult(
                status="fail",
                error_reason="No Planner step in trace. Did the orchestration run the Planner?",
            )

        try:
            planner_parsed = json.loads(planner_outputs[0])
            subtasks = planner_parsed.get("subtasks", []) or []
        except (json.JSONDecodeError, TypeError):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Planner output is not valid JSON with a 'subtasks' array. "
                    f"Got: {planner_outputs[0]!r:.200}"
                ),
            )

        if len(subtasks) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Planner emitted {len(subtasks)} subtasks, expected at least 2. "
                    f"Tighten the Planner instructions to require minimum 2 subtasks."
                ),
            )

        if len(worker_steps) != len(subtasks):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Worker ran {len(worker_steps)} times for {len(subtasks)} subtasks. "
                    f"Sequential orchestration should fan out exactly once per subtask. "
                    f"If counts differ, you may have configured parallel dispatch that "
                    f"double-fires, or the framework version is older than 1.0."
                ),
            )

        for i, step in enumerate(worker_steps, start=1):
            tool_calls = step.tool_calls or []
            web_calls = [tc for tc in tool_calls if getattr(tc, "name", None) == "web_search"]
            if not web_calls:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Worker invocation {i} did not call web_search. The Worker must "
                        f"call the tool on every subtask; tighten the instructions."
                    ),
                )

        if len(writer_steps) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Writer ran {len(writer_steps)} times, expected exactly 1."
                ),
            )

        try:
            writer_parsed = json.loads(writer_steps[0].outputs)
            cited = writer_parsed.get("cited_worker_ids", []) or []
            paragraph = writer_parsed.get("paragraph", "") or ""
        except (json.JSONDecodeError, TypeError):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Writer output is not valid JSON with 'paragraph' and 'cited_worker_ids'. "
                    f"Got: {writer_steps[0].outputs!r:.200}"
                ),
            )

        # Collect worker_ids from the worker step outputs and confirm Writer cited each.
        worker_ids = []
        for step in worker_steps:
            try:
                wp = json.loads(step.outputs)
                wid = wp.get("worker_id")
                if wid:
                    worker_ids.append(wid)
            except (json.JSONDecodeError, TypeError):
                pass
        missing = [w for w in worker_ids if w not in cited]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Writer paragraph did not cite worker_id(s) {missing}. "
                    f"Tighten the Writer instructions to require citing every Worker output."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If `RUN_RESULT.trace` is None, you forgot `capture_trace=True`. If Worker invocation count does not match subtask count, the orchestration is parallel or the framework version is older than 1.0. If Worker has no `web_search` call, tighten the Worker instructions: "Always call web_search before answering."

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `RUN_RESULT = ORCHESTRATION.run(message=RESEARCH_PROMPT, capture_trace=True)`. The orchestration object is already constructed; the seeded research prompt is in `RESEARCH_PROMPT`. The trace inspection happens in the cell after.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (CODE phase):

```python
RUN_RESULT = ORCHESTRATION.run(message=RESEARCH_PROMPT, capture_trace=True)
```

</details>

**Wallet check.** A clean run spends about 8,000 input + 1,600 output tokens across the three agents, which at $0.15 / $0.60 per 1M is roughly $0.0012 + $0.001 = $0.002, or about a fifth of a cent. The Writer is the biggest single line item because its prompt grows with each Worker output. Coffee is still way ahead.

## Task 4: Sum the per-agent token spend and confirm the run stayed under $0.50

The token bill is the cost story for multi-agent. Each agent's context grows: Planner gets the question, Worker gets the question plus a subtask, Writer gets the question plus every Worker output. The compounding is exactly the muscle memory the exam tests.

Build it in this order:

1. From `USAGE_BY_AGENT` (populated in Task 3's OBSERVE phase), sum `prompt_tokens` and `completion_tokens` across every call.
2. Compute total cost at $0.15 per 1M input + $0.60 per 1M output. Confirm it is under $0.50.
3. Print the per-agent breakdown so the cost story is concrete.

The Writer is usually the biggest line item: its input prompt includes the full set of Worker outputs. In production with 20 Workers, the Writer prompt grows linearly, which is why naive multi-agent scaling burns budget fast.

In [ ]:
# NBVAL_SKIP
# Task 4: Sum per-agent token usage and compute the total session cost.

# YOUR CODE: Compute total_input_tokens and total_output_tokens across every
# agent in USAGE_BY_AGENT. Then total_cost_usd at the GPT-4o-mini rates.
# total_input_tokens = sum(...)
# total_output_tokens = sum(...)
# total_cost_usd = ((total_input_tokens / 1_000_000.0) * PRICE_PER_1M_INPUT_USD
#                   + (total_output_tokens / 1_000_000.0) * PRICE_PER_1M_OUTPUT_USD)

print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Total session cost: ${total_cost_usd:.6f}")
print(f"Ceiling: ${COST_CEILING_USD:.2f}")
print()
print("Per-agent breakdown:")
for name in ("Planner", "Worker", "Writer"):
    entries = USAGE_BY_AGENT[name]
    in_sum = sum(e["prompt_tokens"] for e in entries)
    out_sum = sum(e["completion_tokens"] for e in entries)
    cost = (in_sum / 1_000_000.0) * PRICE_PER_1M_INPUT_USD + (out_sum / 1_000_000.0) * PRICE_PER_1M_OUTPUT_USD
    print(f"  {name}: {in_sum} in + {out_sum} out = ${cost:.6f} across {len(entries)} call(s)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Per-agent usage non-zero for all three agents, total cost
# strictly under $0.50, Writer is typically the largest line item but the
# validator does not enforce that ordering (a Planner that fired multiple
# times for retries can flip it).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        for name in ("Planner", "Worker", "Writer"):
            entries = USAGE_BY_AGENT.get(name) or []
            if not entries:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"USAGE_BY_AGENT[{name!r}] is empty. Did Task 3's OBSERVE phase "
                        f"walk every step and capture usage?"
                    ),
                )
            in_sum = sum(e.get("prompt_tokens", 0) for e in entries)
            out_sum = sum(e.get("completion_tokens", 0) for e in entries)
            if in_sum <= 0 or out_sum <= 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{name} usage shows prompt_tokens={in_sum}, completion_tokens={out_sum}. "
                        f"Both must be positive. The framework should expose usage on every "
                        f"step in agent-framework>=1.0.0; if you see zeros, pin the version."
                    ),
                )

        try:
            total_in = total_input_tokens
            total_out = total_output_tokens
            total_cost = total_cost_usd
        except NameError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "total_input_tokens / total_output_tokens / total_cost_usd not defined. "
                    "Compute them in Task 4."
                ),
            )

        if total_cost > COST_CEILING_USD:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total session cost ${total_cost:.6f} exceeds the ${COST_CEILING_USD:.2f} "
                    f"ceiling. Tighten the Planner cap (max 4 subtasks) and shorten Worker "
                    f"summaries."
                ),
            )

        # Cross-check against the manifest's sum to catch arithmetic mistakes.
        expected_in = sum(
            e.get("prompt_tokens", 0)
            for entries in USAGE_BY_AGENT.values()
            for e in entries
        )
        if expected_in != total_in:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"total_input_tokens={total_in} does not match sum of USAGE_BY_AGENT "
                    f"prompt_tokens={expected_in}. Re-check the summation."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If `USAGE_BY_AGENT` shows zeros, the framework version does not expose per-step usage; pin `agent-framework>=1.0.0`. If the cost is above $0.50, the Planner emitted too many subtasks or the Worker summaries are too long; tighten the system prompts.

</details>

<details><summary>Hint 2 (direction)</summary>

Two reductions and one cost formula. `total_input_tokens = sum(e["prompt_tokens"] for entries in USAGE_BY_AGENT.values() for e in entries)`. Same shape for `total_output_tokens` over `completion_tokens`. Cost is `(total_input_tokens / 1_000_000.0) * 0.15 + (total_output_tokens / 1_000_000.0) * 0.60` using the constants already declared in the imports cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
total_input_tokens = sum(
    e["prompt_tokens"]
    for entries in USAGE_BY_AGENT.values()
    for e in entries
)
total_output_tokens = sum(
    e["completion_tokens"]
    for entries in USAGE_BY_AGENT.values()
    for e in entries
)
total_cost_usd = (
    (total_input_tokens / 1_000_000.0) * PRICE_PER_1M_INPUT_USD
    + (total_output_tokens / 1_000_000.0) * PRICE_PER_1M_OUTPUT_USD
)
```

</details>

**Wallet check.** The total session cost in `total_cost_usd` is your real number. Most clean runs land between $0.001 and $0.01. The Writer is the line item to watch; if you scale to 10 Workers, its prompt grows roughly 10x. That is the production lesson encoded in the lab.

## Cleanup

Time to tear it all down. The cell below runs the manifest in reverse-creation order: deployment, project, hub, resource group. The three agents are application-side abstractions and disappear when the kernel exits; the resource group delete is the safety net. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about half a cent for a clean run.** Three agents at GPT-4o-mini rates barely move the meter. The Foundry hub, project, and Standard deployment carry no hourly fee at zero traffic; the resource group delete is the cross-cutting safety net. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. Why does a planner-worker-writer pattern produce better-grounded outputs on open-ended research questions than a single agent given the same prompt and the same tool? Walk through what each agent contributes and what failure mode a single agent has that the three-agent pattern catches.

2. Your team is debating whether to add a Reviewer agent that scores the Writer's output for groundedness and either approves or sends back for revision. What changes in the trace shape, the token cost, and the latency budget? When would you ship the three-agent version versus the four-agent version?

## Exam-Style Practice

**Question 1 (Domain 2, supervisor pattern selection):**

A team needs an agentic system that answers complex customer-support questions by (a) classifying the question into one of three categories, (b) routing it to the right specialist agent for that category, and (c) consolidating the specialist's answer with policy guardrails before returning to the user. Which orchestration pattern fits?

A. Sequential pipeline: Classifier to SpecialistA to SpecialistB to SpecialistC to Guardrail.

B. Supervisor pattern: a Supervisor agent classifies, picks one of three Specialist agents to delegate to, then routes the result through a Guardrail agent.

C. Parallel pattern: all three Specialists run on every question in parallel and a final consolidator picks the best answer.

D. Single-agent: one GPT-4o agent with all three Specialist roles baked into the system prompt, no orchestration framework needed.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: running all three Specialists sequentially on every question wastes tokens and adds latency. The team explicitly wants to ROUTE to ONE specialist.
- B is correct: the supervisor pattern is the canonical multi-agent shape for classify-then-route problems. Microsoft Agent Framework supports it as a first-class orchestration topology. The Guardrail agent is a sequential post-step after the Supervisor returns.
- C is wrong: parallel-then-consolidate is the right pattern for tasks where every Specialist's input is needed; routing problems do not need it.
- D is wrong: baking three Specialist personas into a single GPT-4o prompt confuses the model and produces worse classifications than separating concerns.

</details>

**Question 2 (Domain 2, multi-agent observability):**

A multi-agent customer-support solution emits long conversation traces. The team needs to find the agent step that caused a misrouted-classification failure last Tuesday. The application uses Microsoft Agent Framework 1.0 with traces sent via OpenTelemetry to Application Insights. Which query approach finds the root-cause step fastest?

A. Query App Insights for all `dependencies` rows with `duration > 5s` in the last 7 days; the slow step is the misclassification.

B. Query App Insights for `traces` with `severityLevel >= warning` and `customDimensions.operation_name == "supervisor.classify"` filtered to Tuesday; inspect the input and output of each matching span.

C. Re-run the entire failed conversation from logs and watch the agent steps in the notebook.

D. Disable OpenTelemetry temporarily because the trace volume is too high to query meaningfully.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: latency is not correlated with misclassification. The slow step would be the wrong filter.
- B is correct: OpenTelemetry traces from Agent Framework carry `operation_name` and span attributes that name the agent and the step. Filtering by `supervisor.classify` narrows to the relevant spans. This is the AI-103 documented observability pattern.
- C is wrong: re-running is expensive in tokens and may not reproduce the failure (model behavior can vary). Querying the captured trace is faster and cheaper.
- D is wrong: disabling observability is the wrong direction. You query the volume, you do not discard it.

</details>